# 04 — Classical Computer Vision Module

**Subject:** Computer Vision (Rubén F.)  
**Goal:** Extract visual features from each image and train an RBF-SVM classifier.

## Pipeline
1. Extract three feature sets per image:
   - **HSV colour histogram** (96 dims) — captures affective colour information
   - **Canny edge density** (2 dims) — captures visual complexity
   - **K-means colour palette** (16 dims) — captures dominant colours
2. Concatenate into a single 114-dimensional feature vector
3. Train an **RBF-SVM** classifier
4. Optimise hyperparameters with **Grid Search** (5-fold cross-validation)
5. Save probabilities for the late fusion module

**Expected F1:** ~0.38–0.42 (visual features alone are insufficient without text context)

## Cell 1 — Setup and feature extraction
Load dataset and extract the 114-dimensional feature vector for each image.

In [ ]:
import os
os.chdir("/Users/yesicarb/Desktop/UIE/3º Curso/2 SEM/PROYECTO/emotion/multimodal_emotion")

import pandas as pd
import numpy as np
import sys
sys.path.append("src")
from cv_classic.feature_extractor import extract_features, run_cv_classic

df = pd.read_csv("data/processed/labels.csv")
print(f"Dataset loaded: {df.shape}")
print(df['label'].value_counts())

## Cell 2 — Extract features for all images

This extracts HSV histogram + edge density + k-means colours for every image.
Each image produces a vector of **114 dimensions**:
- HSV histogram: 32 bins × 3 channels = 96
- Edge density: 2
- K-means (k=4): 4 centres × 3 channels + 4 proportions = 16

⏱️ **Expected time: 3–5 minutes** on CPU.

In [ ]:
print("Extracting image features...")
features = []
for i, (_, row) in enumerate(df.iterrows()):
    features.append(extract_features(row['image_path']))
    if i % 500 == 0:
        print(f"  {i}/{len(df)} images processed")

X = np.array(features)
print(f"Feature matrix shape: {X.shape}")  # Expected: (4869, 114)

## Cell 3 — Baseline SVM

Train the SVM with default parameters (C=10, gamma='scale', kernel='rbf').
This gives us a baseline before hyperparameter optimisation.

In [ ]:
svm, scaler, le, y_test, probas = run_cv_classic(df)
print("Baseline SVM done.")

## Cell 4 — Grid Search hyperparameter optimisation

We search over the following parameters:
- **C**: regularisation strength [0.1, 1, 10, 100]
- **gamma**: kernel coefficient ['scale', 'auto', 0.001, 0.01]
- **kernel**: ['rbf', 'linear']

Scoring metric: **macro-averaged F1** (handles class imbalance).  
⏱️ **Expected time: 10–15 minutes** (160 fits with 5-fold CV).

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score
import json

# Encode labels
le = LabelEncoder()
y  = le.fit_transform(df['label'].tolist())

# Same split as all other modules (random_state=42, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Normalise features — SVM is sensitive to feature scale
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# Grid Search
print("Running Grid Search — SVM...")
param_grid = {
    'C':      [0.1, 1, 10, 100],
    'gamma':  ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf', 'linear']
}

gs_svm = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=5, scoring='f1_macro',
    n_jobs=-1, verbose=1
)
gs_svm.fit(X_tr_sc, y_train)

print(f"\nBest parameters: {gs_svm.best_params_}")
print(f"Best F1 (CV):    {gs_svm.best_score_:.4f}")

y_pred_svm = gs_svm.predict(X_te_sc)
probas_svm = gs_svm.predict_proba(X_te_sc)

print("\n=== Classical CV — Optimised SVM ===")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))
print(f"F1 macro (test): {f1_score(y_test, y_pred_svm, average='macro'):.4f}")

# Save results for late fusion
results = {
    'f1_macro':    f1_score(y_test, y_pred_svm, average='macro'),
    'best_params': gs_svm.best_params_,
    'probas':      probas_svm.tolist()
}
with open('results/metrics_cv_classic.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to results/metrics_cv_classic.json")

## Cell 5 — Results analysis

Why is F1 ~0.38? This is expected and justifiable:

- Low-level visual features (colour, edges) do **not capture emotional semantics**.
- A photo of a beach can express positive or neutral emotion depending on context.
- **This result motivates the use of ResNet18 and multimodal fusion** in the next modules.

Key observation: the model still outperforms the random baseline (0.33), confirming
that visual information carries *some* signal, just not enough on its own.

In [ ]:
import matplotlib.pyplot as plt

modules   = ['Random\nBaseline', 'CV Classic\n(SVM)']
f1_scores = [0.33, f1_score(y_test, y_pred_svm, average='macro')]
colors    = ['#9ca3af', '#f97316']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(modules, f1_scores, color=colors, width=0.4, edgecolor='white')
for bar, val in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 0.8)
ax.axhline(0.33, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax.set_ylabel('F1-score (macro)')
ax.set_title('Classical CV Module — Results')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/cv_classic_results.png', dpi=150)
plt.show()
print("Figure saved to results/figures/cv_classic_results.png")